# Notebook 18: LangGraph - Building Your First Workflow

**🎯 Goal:** To learn how to build and control the flow of your LangGraph applications, with a focus on conditional routing and effective state management.

## 🧩 Concept Overview

In the previous notebook, we learned about the core components: State, Nodes, and Edges. Now, let's focus on how to assemble them into useful workflows.

The power of LangGraph comes from its ability to **dynamically route** the workflow based on the current state. This allows you to build agents that can make decisions, use tools, and loop until a task is complete.

### 3.1. Simple Sequential Workflow (Recap)

The simplest workflow is a straight line. This is great for tasks that have a fixed sequence of steps, like `research -> summarize -> review`.

In [ ]:
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, END

class SequentialState(TypedDict):
    topic: str
    research: str
    summary: str

def research_node(state: SequentialState) -> dict:
    print("--- Researching --- ")
    return {"research": f"Data on {state['topic']}."}

def summarize_node(state: SequentialState) -> dict:
    print("--- Summarizing ---")
    return {"summary": f"Summary of {state['research']}"}

def review_node(state: SequentialState) -> dict:
    print("--- Reviewing ---")
    # In a real app, this might add review comments
    return {}

workflow = StateGraph(SequentialState)
workflow.add_node("researcher", research_node)
workflow.add_node("summarizer", summarize_node)
workflow.add_node("reviewer", review_node)

workflow.set_entry_point("researcher")
workflow.add_edge("researcher", "summarizer")
workflow.add_edge("summarizer", "reviewer")
workflow.add_edge("reviewer", END)

app = workflow.compile()
app.invoke({"topic": "AI agents"})

### 3.2. Conditional Routing

This is where LangGraph truly shines. Using `add_conditional_edges`, you can create a 'router' node that decides which path to take based on the current state.

Let's build a simple agent that decides whether it needs to use a tool to answer a question.

In [ ]:
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv

load_dotenv()
llm = ChatOpenAI(model="gpt-4o")

class AgentState(TypedDict):
    messages: Annotated[list, operator.add]

# Node that calls the LLM
def call_llm(state: AgentState) -> dict:
    print("--- Calling LLM ---")
    response = llm.invoke(state['messages'])
    return {"messages": [response]}

# Node that simulates a tool call
def use_tool(state: AgentState) -> dict:
    print("--- Using Tool ---")
    # In a real app, this would execute a tool based on the LLM's request
    return {"messages": [AIMessage(content="The weather in SF is 65°F.")]}

# The conditional router function
def should_use_tool(state: AgentState) -> str:
    print("--- Checking for tool use ---")
    last_message = state['messages'][-1]
    # A simple heuristic: if the LLM mentions 'tool' or 'search', route to the tool
    if "tool" in last_message.content.lower() or "search" in last_message.content.lower():
        return "use_tool"
    else:
        return "end"

cond_workflow = StateGraph(AgentState)
cond_workflow.add_node("agent", call_llm)
cond_workflow.add_node("tool", use_tool)

cond_workflow.set_entry_point("agent")

# Add the conditional edge
cond_workflow.add_conditional_edges(
    "agent",
    should_use_tool,
    {
        "use_tool": "tool",
        "end": END
    }
)
cond_workflow.add_edge("tool", END)

app = cond_workflow.compile()


In [ ]:
# Example 1: The LLM decides it does NOT need a tool
print("--- Running Example 1 ---")
inputs = {"messages": [HumanMessage(content="What is the capital of France?")]}
result = app.invoke(inputs)
print(f"Final Answer: {result['messages'][-1].content}")

# Example 2: The LLM decides it DOES need a tool (we'll mock the LLM's response)
print("\n--- Running Example 2 ---")
inputs = {"messages": [AIMessage(content="I should use a tool to find the weather.")]}
result = app.invoke(inputs)
print(f"Final Answer: {result['messages'][-1].content}")

### 3.3. State Design Patterns

How you design your state is crucial for building robust agents.

- **Accumulating Messages:** Use `Annotated[list, operator.add]` to build a conversation history. This is the most common pattern.
- **Tracking Metadata:** Add fields to your state to track things like the number of steps, which agent was the last one to act, or error counts. This is essential for controlling loops and preventing your agent from getting stuck.

In [ ]:
class LoopingState(TypedDict):
    messages: Annotated[list, operator.add]
    # Metadata to control the loop
    step_count: int

def agent_step(state: LoopingState) -> dict:
    print(f"--- Step {state['step_count']} ---")
    # In a real app, the LLM would do something here
    return {"messages": [AIMessage(content="I'm still working...")], "step_count": state['step_count'] + 1}

def should_continue(state: LoopingState) -> str:
    # If we've taken too many steps, exit the loop
    if state['step_count'] > 3:
        return "end"
    return "continue"

loop_workflow = StateGraph(LoopingState)
loop_workflow.add_node("agent", agent_step)
loop_workflow.set_entry_point("agent")
loop_workflow.add_conditional_edges("agent", should_continue, {"continue": "agent", "end": END})
loop_app = loop_workflow.compile()

# Run the looping app
result = loop_app.invoke({"messages": [], "step_count": 0})
print("\n--- Loop Finished ---")
print(result['messages'][-1].content)

## 🏁 Summary

You can now build workflows that go beyond simple sequences.

1.  **Conditional Edges are for Control:** Use `add_conditional_edges` to route your workflow based on the agent's state, enabling decision-making.
2.  **State is for Memory and Control:** A well-designed state doesn't just hold data; it holds metadata that you can use to control the flow of your graph, especially for loops.

In the next notebook, we'll dive into building teams of collaborating agents.